---
title: "Introduction to Predictive Modeling with Pipelines"
jupyter: python3
---

# Data Context 

The data for today is taken from [this Kaggle dataset](https://www.kaggle.com/datasets/xavya77/nhl04to18) of NHL player
statistics. The data contain statistics on every NHL player for the 
2004 through 2018 seasons assembled by Xavya. 

- Data were scraped from: <https://www.hockey-reference.com/> 
- Skater statistics were scraped from: <https://www.hockey-reference.com/leagues/NHL_2018_skaters.html>
- Hart MVP voting was scraped from:
<https://www.hockey-reference.com/awards/hart.html>

We also added in [this Tidy Tuesday dataset](https://github.com/rfordatascience/tidytuesday/blob/main/data/2024/2024-01-09/readme.md) of player height, weight, and birthday information.

This information was acquired through the [NHL API](https://github.com/Zmalski/NHL-API-Reference). 

## Loading the Data 

I've saved the `.csv` to Dropbox, so we can read it in using a URL. Run the code
provided to load the data.

In [ ]:
#| label: load-data-Python
#| echo: true
#| code-line-numbers: false

import pandas as pd

data_nhl = pd.read_csv("https://www.dropbox.com/scl/fi/bl8fc27dxunn585o6tyhx/nhl_player_stats.csv?rlkey=kl9usgrmm0ci25n0v95irlzc3&st=27c9lurl&dl=1")

## Inspecting the Data

Take a few minutes to look over the data. 

- What is the **observational unit**? (i.e., what uniquely identifies a row?)

- What are the variables?
    * Are there any variables whose meaning is not obvious?

- What are the data types of the variables (e.g., number, character, date)

## Observations & Variables 

## Observations & Variables 

Each row is a player's statistics from one `season`. The variables measured
about each player are:

* identification: `first_name`, `last_name`
* physicals: `height`, `weight`, `age` (for that season)
* birth Information: `birth_date`, `birth_day`, `birth_month`, `birth_city`,
`birth_country`, `birth_state_province`
* position & team: `team`, `position_code` (right wing, left wing, center, 
or defense), `position_type` (forward or defense), and `sweater_number`.
* statistics for the season: `games_played`, `goals`, `assists`, `points`
(goals plus assists), `plusminus` (team goal differential when player is
playing), and `penalty_minutes`.

## Posing a Question

We will use these data to try to address this question: 

> What features of an NHL player are associated with them scoring more points
> (goals + assists) in a particular season?

# Steps 1 & 2: Cleaning & Visualizing the Data

Our goal here is to visualize variables we are interested in to look for 
interesting patterns, **while also** looking for issues or anomalies to fix in
the data.

## Numeric Variables

What are we looking for?

-   **Unusual observations** that we might need to *remove*.

-   Variables that are very **skewed** and might need to be *transformed*.

-   Variables that are extremely **multimodal** and might need to be *binned*.

-   Variables with values that we might want to **omit** from our study because
they are *not relevant* to the question.

### Investigating with Visuals

In [ ]:
from plotnine import *

Use the template code provided to create visuals of each numeric variable in the
`data_nhl` dataset: 

- `height_in_inches`
- `weight_in_pounds`
- `age`
- `birth_month` 
- `games_played`
- `goals`
- `assists`
- `points`
- `plusminus`
- `penalty_minutes`

Do you find anything notable?

In [ ]:
#| label: plotnine-template

(
  ggplot(data = data_nhl, 
       mapping = aes(x = "points")
       ) +
  geom_histogram()
)

### Cleaning the Data

Based on your investigation, you will likely want to filter the data to 
exclude certain observations. Fill in the code below to filter the data (based
on Dr. T's slides):

*Replace the `___` with the relational statement you want (e.g., `> 50`)!*

In [ ]:
#| label: data-cleaning-numerical

dat_nhl_clean = dat_nhl[
    (dat_nhl["games_played"] ___) &
    (dat_nhl["points"] ___) &
    (dat_nhl["age"] ___) &
    (dat_nhl["penalty_minutes"] ___)
]

## Categorical variables

What are we looking for?

-   Categories that should be **combined** to make a few larger ones.
-   Variables that should have an `"Other"` category to combine the rarest
categories.
-  Variables with **too many categories** to reasonably include them in our
analysis.
- Variables that are **subsets** of each other, so we might want to include only
one or the other.
- Anything that suggests **issues with the data** to fix.

### Investigating with Visuals

Use the template code provided to create visuals of each numeric variable in the
`data_nhl` dataset: 

- `season`
- `team`
- `position_code`,
- `position_type`
- `birth_city`
- `birth_country`
- `birth_state_province`

Do you find anything notable?

In [ ]:
(
  ggplot(data_nhl, 
       mapping = aes(x = "position_code")) +
  geom_bar()
  )

### Cleaning the Data

Based on your investigation, you will likely want to filter the data to 
exclude certain observations. Fill in the code below to filter the data (based
on Dr. T's slides):

In [ ]:
#| label: data-cleaning-categorical

dat_nhl = dat_nhl.drop_duplicates(
  subset = []
  )

top_countries = (
  dat_nhl[]
  .value_counts()
  .nlargest()
  .index
  )

dat_nhl["can_or_usa"] = (
  dat_nhl[]
  .where(dat_nhl[].isin(top_countries), 
         "Other")
  )

### Check in

Make sure your re-visualize your variables after making the cleaning changes, to
ensure that this accomplished what you hoped!

Variables to Check:

- `points`
- `penalty_minutes`
- `games_played`
- `season` 
- `can_or_usa`

In [ ]:
#| label: cleaning-check

# Step 3: Choosing Candidate Predictors

We will answer this question with a *predictive model*, that uses information in
some *predictor* variables to create a process for guessing the value of the
*target* variable (`points`).

To decide which variables are worth including in our eventual model, we need to
understand how they associate with the target variable of `points`.

## Numeric Variables

Consider the *numeric* variables in this dataset.  Which ones might be
reasonable to use in our model?

Use the code provided to visualize the relationship between `points` and a few
of these potential predictors. Decide which variable(s) seem to have the 
strongest relationship with our target variable (`points`). 

In [ ]:
#| label: ggplot-template-two-numerical

(
  ggplot(data = data_nhl, 
       mapping = aes(x = "age", y = "points")
       ) +
  geom_point() +
  geom_smooth()
)

## Categorical Variables

Consider the *categorical* variables in this dataset.  Which ones might be
reasonable to use in our model?

Use the code provided to visualize the relationship between `points` and a few
of these potential predictors. Decide which variable(s) seem to have the 
strongest relationship with our target variable (`points`). 

```{bash}
# Note you will need to install the ridgenine package for this plot to work!
pip install ridgenine
```

In [ ]:
#| label: ggplot-template-one-cat-one-num

from ridgenine import geom_density_ridges

(
  ggplot(data = data_nhl, 
         mapping = aes(x = "points", y = "position_type")
         ) +
  geom_density_ridges() 
)

## Multiple Variables

Sometimes, the way one variable associates with `points` might be different
depending on another variables.

It is believed that players who are born earlier in the month are more likely to
become professional athletes, because they are on the older end of the league
for their age growing up.  

It may be that this trend only applies in some countries, since not all
countries have the same youth sports system.

Let's see!

In [ ]:
#| label: points-by-birth-faceted-by-country

(
  ggplot(data = data_nhl, 
         mapping = aes(x = "points", y = "can_or_usa")
         ) +
  geom_density_ridges() +
  facet_wrap("birth_month") 
)

### Make Predictor Recipes

Now, we are ready to prepare our **recipes**: the different combinations of
predictors that we think could produce a good model.

We'll try two recipes today:

* One with *all* the possible predictors

* One with only the three "best" predictors that you choose.

In [ ]:
vars_all = ["height", "weight", "age", "can_or_usa", "birth_month", 
            "position_code", "season", "team", "penalty_minutes"]

vars_best = []

## Specifying Candidate Models

The other decision we must make  is which *model types* to try.

In this analysis, we will try *Linear Regression* and a *Decision Tree*. 

- Linear regression is a natural choice since we have a continuous numerical
outcome and the interpretations are quite simple. 

- A decision tree is a simple choice for a non-statistical model that also has
easy interpretations. 

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

lr_spec = LinearRegression()

dt_spec = DecisionTreeRegressor(____)

## Pre-Processing Options

Recall that we noticed a few interesting things in our exploratory data
analysis:

* `penalty_minutes` is very right-skewed 

* Birth country seems to impact how birth month impacts the target. 

We will add a *tranformation* and an *interaction* to our workflow.

In [ ]:
#| label: make-general-transformers

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, PolynomialFeatures

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

log_transformer = Pipeline([
    ("log", FunctionTransformer(np.log1p, validate = True))
])

interact_transformer = Pipeline([
  ("onehot", OneHotEncoder(drop = "first")),
  ("interact", PolynomialFeatures(interaction_only = True, include_bias = True))
])

In [ ]:
#| label: use-transformers-on-specific-columns

rec_all = ColumnTransformer([
    ("cat", categorical_transformer, ["position_code", "season", "team"]),
    ("int", interact_transformer, ["can_or_usa", "birth_month"]),
    ("log", log_transformer, ["penalty_minutes"]),
    ("num", "passthrough", ["age", "height", "weight"])
])

rec_best = ColumnTransformer([
    ("cat", categorical_transformer, ["position_type", "season", "team"])
])

### Combining Data & Models

Finally, we will try all our different **model type** options with all our
different **recipe and preprocessing** choices.

Each of these candidate options is called a **workflow** or a **pipeline**.

In [ ]:
#| label: specify-model-and-transformers

from sklearn.pipeline import make_pipeline
from sklearn.base import clone

# Linear regression with all predictors
wflow_lr_all = make_pipeline(clone(rec_all), clone(lr_spec))

# Linear regression with best predictors
wflow_lr_best = make_pipeline(clone(rec_best), clone(lr_spec))

# Decision tree with all predictors
wflow_dt_all = make_pipeline(clone(rec_all), clone(dt_spec))

# Decision tree with best predictors
wflow_dt_best = make_pipeline(clone(rec_best), clone(dt_spec))

# Step 4 -- Choosing between Competing Models

Whew! All this work, and we haven't even fit a model yet!  

Everything we have done to this point was *setup*: establishing our plans for a
few different workflows to try.

Now, how will we compare the four options and see which one is most successful
at predicting `points`?  

The process goes like this:

1. Randomly set aside 20% of our data rows as a *test set*.  The rest is
called the *training set*.

2. Fit all of the workflows on the *training set*.

3. See how well each fitted workflow does at predicting on the *test set*.

## Test / Train Split

In [ ]:
#| label: test-train-split

from sklearn.model_selection import train_test_split

# Same random state (seed) as Python users get consistent results
data_nhl_train, data_nhl_test = train_test_split(data_nhl, 
                                                 test_size = __, 
                                                 random_state = ____)

### Fit Our Models on the Training Data

In [ ]:
#| label: fit-models-to-training-data

# Declare predictor variables
X_train = data_nhl_train.drop(columns = ["points"])
# Declare target variable
y_train = data_nhl_train["points"]

# Fit the workflows
wflow_lr_all_fit = wflow_lr_all.fit(X_train, y_train)
wflow_lr_best_fit = wflow_lr_best.fit(X_train, y_train)
wflow_dt_all_fit = wflow_dt_all.fit(X_train, y_train)
wflow_dt_best_fit = wflow_dt_best.fit(X_train, y_train)

## Assess Models on Test Data

Finally, we will use these fitted models to make predictions on the test data, 
and we'll see how close these predictions were to the true observed values for
`points`

In this analysis, we are using *root mean squared error* as our **metric** for
the test data.  That is, we take the squared difference between the *predicted* 
number of points a player will get and the *actual* number of points they get, 
then we add those numbers up and take the average. 

$$ \sqrt \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{n}$$

In [ ]:
#| label: predictions-for-test-data
#| code-line-numbers: false

from sklearn.metrics import mean_squared_error

# Declare predictor variables
X_test = data_nhl_test.drop(columns = ["points"])
# Declare target variable
y_test = data_nhl_test["points"]

# Make predictions
pred_lr_all = wflow_lr_all_fit.predict(X_test)
pred_lr_best = wflow_lr_best_fit.predict(X_test)
pred_dt_all = wflow_dt_all_fit.predict(X_test)
pred_dt_best = wflow_dt_best_fit.predict(X_test)

# Calculate prediction errors
rmse_lr_all = np.sqrt(mean_squared_error(y_test, pred_lr_all))
rmse_lr_best = np.sqrt(mean_squared_error(y_test, pred_lr_best))
rmse_dt_all = np.sqrt(mean_squared_error(y_test, pred_dt_all))
rmse_dt_best = np.sqrt(mean_squared_error(y_test, pred_dt_best))

print("RMSE (Linear All):", rmse_lr_all)
print("RMSE (Linear Best):", rmse_lr_best)
print("RMSE (Tree All):", rmse_dt_all)
print("RMSE (Tree Best):", rmse_dt_best)

## What did you get???

Of our four candidate workflows, which model is the "best" model? 

By how much? How close is the next best model?

## Fit your final model

We used out test/training split to **assess our candidates**.  However, once
we've chosen a winner, we want to make sure we use *all our data* for our
final conclusions!

So, we have one last model fitting task:

In [ ]:
#| label: fit-final-model-python

# Declare predictor variables
X_full = data_nhl.drop(columns = ["points"])
# Declare target variable
y_full = data_nhl["points"]

# Fit your chosen pipeline to the full data!
final_model = _______.fit(X_full, y_full)

# Step 5 -- Interpreting Model Conclusions

- Sometimes, the goal of your machine learning process is simply to **produce a 
model**. 
    * You can use the model to predict which players you should draft!

- Other times, the goal is more about **interpretation**. 
    * You want to tell stories about the patterns in the data.

Since our best final model was a *decision tree*, we'll make a 
**dendrogram** visualizing the splits. 

```{bash}
pip install dtreeviz
```

In [ ]:
import dtreeviz

viz = dtreeviz.model(
    final_model.named_steps["decisiontreeregressor"],
    X_train=final_model.named_steps["columntransformer"].transform(X_train),
    y_train=y_train,
    target_name="target",
    feature_names=[f"feature_{i}" for i in range(n_features)]
)
viz.view()

In [ ]:
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(
    final_model.named_steps["decisiontreeregressor"],
    max_depth=3,
    filled=True,
    rounded=True
)
graphviz.Source(dot_data).view()

In [ ]:
#| label: dendrogram-of-decision-tree-model
#| eval: false

from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

n_features = final_model.named_steps["decisiontreeregressor"].n_features_in_
feature_names = [f"feature_{i}" for i in range(n_features)]

plt.figure(figsize=(16, 8))
plot_tree(
    final_model.named_steps["decisiontreeregressor"],
    feature_names=feature_names,
    filled=True,
    rounded=True,
    max_depth=3
)

plt.show()

The plot references the names of the features which we can track down here:

In [ ]:
final_model.named_steps["columntransformer"].transformers_

## Models that can do harm

- Predictive models are great at **predicting**. 
- Predictive models are not great in knowing if their model is **harmful**. 

Suppose instead we wanted to predict a student's likelihood of success in
college. Our predictive model suggests that: 

- `zip_code`, 
- `parental_education`, 
- `first_generation`
- `standardized_exam_score` 

are strongly related to students' college performance. 

**Who does this model privilege? Who is being harmed by this model?**
